In [ ]:
import test_pipeline
import slice_util
import os
import numpy as np
import glob
import pandas as pd
import shutil

In [ ]:
config_dir = '/home/jhahn/puzzlefusion-plusplus/config'
files_root =  '/home/jhahn/puzzlefusion-plusplus/web/files'
ckpt_path= '/home/jhahn/puzzlefusion-plusplus/brain_lightsheet/denoiser/everyday_2000epoch/training/last.ckpt'

In [ ]:
import importlib
import puzzlefusion_plusplus.denoiser.dataset.dataset
importlib.reload(puzzlefusion_plusplus.denoiser.dataset.dataset)
from puzzlefusion_plusplus.denoiser.dataset.dataset import build_test_dataloader
import test_pipeline
importlib.reload(test_pipeline)

cfg = test_pipeline.load_cfg(config_dir)

num_of_missing_slices = 0
from_index = 100
tickness = 0.005

data_ids = [f'sliced_on_1_0_0_{tickness:.3f}_True_{num_of_missing_slices}_{from_index}_{True}_700']#['0408']


tiff_dir_root, obj_dir_root, pc_dir_root, inference_dir_root, render_output_dir = test_pipeline.init_dir(files_root,data_ids)
print('tiff_dir_root',tiff_dir_root)
print('obj_dir_root',obj_dir_root)
print('pc_dir_root',pc_dir_root)
print('inference_dir_root',inference_dir_root)
print('render_output_dir',render_output_dir)

tiff_dir = f'/data/jhahn/data/brain_lightsheet/slices/{"_".join(data_ids[0].split("_")[:5])}'
tiff_list = os.listdir(tiff_dir)
tiff_list.sort(key = lambda x: int(x.split(".")[0]))
tiff_list = tiff_list[from_index:]

for _i, f in enumerate(tiff_list):
    _id = int(f.split(".")[-2])
    if _i % (num_of_missing_slices+1) == 0:
        shutil.copyfile(tiff_dir + "/" + f, tiff_dir_root + "/"+ f )    
    #if _i > 10:
    #    break
'''
glb_dir = f'/data/jhahn/data/shape_dataset/data/brain_lightsheet/{data_ids[0]}/fractured_0'
for f in os.listdir(glb_dir):
    _id = int(f.split(".")[-2])

    shutil.copyfile(f'{tiff_dir}/{_id}.tif', f'{tiff_dir_root}/{_id}.tif')    
'''




In [4]:

import trimesh
import importlib
import test_pipeline
importlib.reload(test_pipeline)
import test_pipeline
no_gap_between_slices = True
is_curvature = False
num_of_missing_slices = 5
obj_dir_list_relative = test_pipeline.tiff_2_obj(cfg, tiff_dir_root, tickness, obj_dir_root, pc_dir_root, num_of_missing_slices, no_gap_between_slices, is_curvature)
#obj_dir_list_relative = ['0.001_0.007']
#obj_dir_list_relative

obj_files = []
for _o in os.listdir(obj_dir_root + "/"+obj_dir_list_relative[0]):
    if not _o.endswith(".glb"):
        continue
    _glb = trimesh.load(obj_dir_root + "/"+obj_dir_list_relative[0]+"/"+_o)
    _new_obj_filename = render_output_dir+"/objs/"+_o.replace(".glb",".obj")
    _glb.export(_new_obj_filename)
    #print(_new_obj_filename)
    obj_files.append(_new_obj_filename)
slice_util.combine_obj_files(obj_files, render_output_dir+"/original.obj")

dataset ['test']
save_path /home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.005_True_0_100_True_700/pc


Processing test data:   0%|          | 0/1 [00:00<?, ?it/s]

/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.005_True_0_100_True_700/objs/test/fractured_0 ['100.glb', '106.glb', '112.glb', '118.glb', '124.glb', '130.glb', '136.glb', '142.glb', '148.glb', '154.glb', '160.glb', '166.glb', '172.glb', '178.glb', '184.glb', '190.glb', '196.glb', '202.glb']


Processing test data: 100%|██████████| 1/1 [00:46<00:00, 46.79s/it]

test/fractured_0


[]

In [5]:
import test_pipeline
importlib.reload(test_pipeline)
test_pipeline.inference(cfg, pc_dir_root, obj_dir_list_relative, ckpt_path, inference_dir_root)
import importlib
import render_inference_result
importlib.reload(slice_util)
importlib.reload(render_inference_result)
import puzzlefusion_plusplus.denoiser.dataset.dataset
importlib.reload(puzzlefusion_plusplus.denoiser.dataset.dataset)
from puzzlefusion_plusplus.denoiser.dataset.dataset import build_test_dataloader


vertices_gt, obj_id_list = render_inference_result.get_vertices(inference_dir_root, obj_dir_root, device=test_pipeline.device)


{'hydra': {'output_subdir': None, 'run': {'dir': '.'}}, 'defaults': ['_self_', 'denoiser/data', 'denoiser/model', 'denoiser/encoder', 'verifier/model', 'ae/model', 'ae/vq_vae', {'override hydra/hydra_logging': 'disabled'}, {'override hydra/job_logging': 'disabled'}], 'denoiser': {'ckpt_path': '/home/jhahn/puzzlefusion-plusplus/brain_lightsheet/denoiser/everyday_2000epoch/training/last.ckpt', 'data': {'val_batch_size': 1, 'matching_data_path': './data/matching_data/', 'batch_size': 64, 'num_workers': 64, 'data_fn': 'brain_lightsheet.{}.txt', 'data_dir': '/data/jhahn/data/shape_dataset/pc_data/brain_lightsheet/train/', 'data_val_dir': '/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.005_True_0_100_True_700/pc', 'mesh_data_dir': '/data/jhahn/data/shape_dataset/data', 'rot_range': -1, 'overfit': -1, 'min_num_part': 2, 'max_num_part': 20}, 'ae': {'ae_name': {'_target_': 'puzzlefusion_plusplus.denoiser.model.modules.encoder.VQVAE'}, 'n_embeddings': 1024, 'embedding_dim': 16, 'n

100%|██████████| 1/1 [00:00<00:00, 393.68it/s]
/home/jhahn/puzzlefusion-plusplus/test_pipeline.py:206: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  denoiser_weights = torch

Testing: |          | 0/? [00:00<?, ?it/s]

_save_inference_data /home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.005_True_0_100_True_700/inference/0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      eval/part_acc         0.0555555559694767
       eval/rmse_r          10.328568458557129
       eval/rmse_t          0.12339871376752853
      eval/shape_cd         0.04124872386455536
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


100%|██████████| 18/18 [00:00<00:00, 189.30it/s]


In [6]:
#test_pipeline.inference(cfg, pc_dir_root, obj_dir_list_relative, ckpt_path, inference_dir_root)
import importlib
import render_inference_result
importlib.reload(slice_util)
importlib.reload(render_inference_result)
import puzzlefusion_plusplus.denoiser.dataset.dataset
importlib.reload(puzzlefusion_plusplus.denoiser.dataset.dataset)
from puzzlefusion_plusplus.denoiser.dataset.dataset import build_test_dataloader
import test_pipeline
importlib.reload(test_pipeline)

test_pipeline.render(inference_dir_root,obj_id_list , vertices_gt, render_output_dir)
#shape_cd = test_pipeline.eval( vertices_gt,inference_dir_root,render_output_dir)
#shape_cd

gen snapshot images:   0%|          | 0/18 [00:00<?, ?it/s]/home/jhahn/puzzlefusion-plusplus/2d_2_pcd/render_inference_result.py:162: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  tr = Translate(torch.FloatTensor([init_trans]), dtype=torch.float32, device=device)
gen snapshot images: 100%|██████████| 18/18 [00:01<00:00, 11.99it/s]


predict_0 (20, 18, 7)


rotate by dice score: 2it [00:01,  1.06it/s]

1, 0.045, 0.121, 332


rotate by dice score: 3it [00:03,  1.32s/it]

...

rotate by dice score: 17it [00:29,  1.85s/it]

16, 0.041, 0.206, 343


rotate by dice score: 18it [00:31,  1.76s/it]


17, 0.029, 0.205, 42


gen pngs: 100%|██████████| 38/38 [00:28<00:00,  1.33it/s]
